In [ ]:
from processing.satellites import gee_collections, get_bands, get_mean_value, process_collection, plot_time_series, regenerate_all_plots, avdc_collections, download_avdc_omi_station_data

In [ ]:
collections_not_processed, errors = regenerate_all_plots(gee_collections=gee_collections)
collections_not_processed

In [ ]:
collections_not_processed = [
    # 'ESA/CCI/FireCCI/5_1',
    # 'COPERNICUS/S5P/OFFL/L3_CO',
    # 'FIRMS',
    # 'JAXA/GCOM-C/L3/LAND/LST/V3',
    # 'JAXA/GPM_L3/GSMaP/v8/operational',
    # 'MODIS/061/MOD14A1',
    # 'MODIS/061/MYD14A1',
    ]

# process specified collections
collections_subset = collections_not_processed
# collections_subset = ['MODIS/061/MOD08_M3', 'TRMM/3B42']
for collection in gee_collections.keys():
    if collection in collections_subset:
        for variable in gee_collections[collection]['variables']:
            process_collection(collection=collection, variable=variable, verbosity=1)

In [ ]:
# # process all available collections
# # NB: This will take several hours or even days to complete!
# for collection in gee_collections.keys():
#     for variable in gee_collections[collection]['variables']:
#         process_collection()

In [ ]:
# for collection in ['MODIS/061/MOD08_M3']:
#     display(collection, get_bands(collection))

In [ ]:
collection = 'OMI/V03/L2OVP/OMTO3'
product = 'aura_omi_l2ovp_omto3_v8.5_nairobi_175.txt'
df, metadata = download_avdc_omi_station_data(collection=collection, product=product)
display(metadata)
df.describe()


In [ ]:
import polars as pl
import matplotlib.pyplot as plt

# aggregate to daily values
df_1d_avdc = df.group_by('dte', maintain_order=True).agg([
    pl.col('Ozone').mean(),
    # Add more aggregations as needed, for example:
    # pl.col('OtherColumn').mean().alias('MeanOtherColumn'),
    # pl.col('OtherColumn').count().alias('CountOtherColumn')
])
df_1d_avdc.schema
# plt.plot(df_1d_avdc['dte'], df_1d_avdc['Ozone'])

In [ ]:
df_s5p_offl_l3_o3 = pl.read_parquet('data/level3/copernicus/s5p/offl/l3_o3/O3_column_number_density.parquet')
df_s5p_offl_l3_o3 = df_s5p_offl_l3_o3.with_columns(pl.col('dte').dt.date())
# remove null values
df_s5p_offl_l3_o3 = df_s5p_offl_l3_o3.filter(pl.col('nrb') > 0)
df_s5p_offl_l3_o3.describe()
df_s5p_offl_l3_o3.schema

In [ ]:
df_toms = pl.read_parquet('data/level3/toms/merged/ozone.parquet')
df_toms = df_toms.with_columns(pl.col('dte').dt.date())
# remove null values
df_toms = df_toms.filter(pl.col('nrb') > 0)
df_toms.describe()

In [ ]:
# unit conversion DU=mol/m2×(2.687×10206.022×1023​)≈mol/m2×2241.15
cf = 2241.15

In [ ]:
fig = plt.figure(figsize=(10, 7))
plt.plot(df_toms['dte'], df_toms['nrb'], c="orange", label='TOMS merged [DU]')
plt.plot(df_1d_avdc['dte'], df_1d_avdc['Ozone'], label='AVDC EOS Aura OMI OMTO3 (v8.5, Collection 3) [DU]')
plt.plot(df_s5p_offl_l3_o3['dte'], df_s5p_offl_l3_o3['nrb']*cf, c="r", label=f'Copernicus S5P offl l3 O3 [mol/m2 x {cf}]')
plt.legend()
plt.title('NRB Total column ozone - daily values')
plt.ylabel ('Ozone (DU)')
plt.show()
fig.savefig('results/satellites/aura_vs_s5p_vs_toms_timeseries.png')

In [ ]:
# combine data frames
df_aura_toms = df_toms.join(df_1d_avdc, on='dte', how='inner')
df_aura_toms.drop_in_place('mkn')
df_aura_toms = df_aura_toms.rename({'nrb': 'tco_toms', 'Ozone': 'tco_aura'})
df_aura_toms.write_parquet('results/satellites/aura_vs_toms.parquet')
print(df_aura_toms.describe())

df_s5p_toms = df_toms.join(df_s5p_offl_l3_o3, on='dte', how='inner')
df_s5p_toms.drop_in_place('mkn')
df_s5p_toms.drop_in_place('mkn_right')
df_s5p_toms = df_s5p_toms.rename({'nrb': 'tco_toms', 'nrb_right': 'tco_s5p'})
df_s5p_toms.write_parquet('results/satellites/s5p_vs_toms.parquet')
print(df_s5p_toms.describe())

fig = plt.figure(figsize=(7, 7))
plt.scatter(df_aura_toms['tco_toms'], df_aura_toms['tco_aura'], c="orange", s=8, label=f'AVDC vs TOMS [DU], n={len(df_aura_toms)}')
plt.scatter(df_s5p_toms['tco_toms'], df_s5p_toms['tco_s5p'] * cf, s=8, label=f'S5P [mol/m2 x {cf}] vs TOMS [DU], n={len(df_s5p_toms)}')
plt.legend()
plt.title('NRB Total column ozone - daily values')
plt.xlabel('Ozone [DU]')
plt.ylabel('Ozone [DU]')
plt.show()
fig.savefig('results/satellites/aura_vs_s5p_vs_toms_correlation.png')